In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
import pandas as pd
import numpy as np
import os

In [ ]:
strategy = tf.distribute.MirroredStrategy()

print("Number of GPUs:", strategy.num_replicas_in_sync)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2

def build_cnn():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        BatchNormalization(),
        MaxPooling2D(),

        Conv2D(64, (3,3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D(),

        Flatten(),
        Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ])
    return model

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

def build_mobilenet():
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False  # Freeze

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=base.input, outputs=out)

In [ ]:
from tensorflow.keras.applications import ResNet50

def build_resnet():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=base.input, outputs=out)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

def build_efficientnet():
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = Dropout(0.5)(x)
    out = Dense(1, activation='sigmoid')(x)

    return Model(inputs=base.input, outputs=out)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
def train_model(build_fn, name):
    
    with strategy.scope():
        model = build_fn()

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=8,
        callbacks=[early_stop],
        verbose=1
    )

    val_acc = max(history.history['val_accuracy'])
    print(f"{name} Best Val Accuracy:", val_acc)

    return model, val_acc ,history

In [ ]:
results = {}

cnn_model, acc_cnn, history_cnn = train_model(build_cnn, "CNN")
results["CNN"] = (cnn_model, acc_cnn)

mobilenet_model, acc_mob, history_mobilenet = train_model(build_mobilenet, "MobileNet")
results["MobileNet"] = (mobilenet_model, acc_mob)

resnet_model, acc_res, history_resnet = train_model(build_resnet, "ResNet")
results["ResNet"] = (resnet_model, acc_res)

efficient_model, acc_eff, history_efficientnet = train_model(build_efficientnet, "EfficientNet")
results["EfficientNet"] = (efficient_model, acc_eff)

In [ ]:
best_name = max(results, key=lambda x: results[x][1])
best_model = results[best_name][0]

print("Best Model:", best_name)

In [ ]:
import matplotlib.pyplot as plt

models = list(results.keys())
accuracies = [results[m][1] for m in models]
#Itz Ary99Pk
plt.bar(models, accuracies)
plt.ylabel("Validation Accuracy")
plt.title("Model Comparison")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_cnn.history['val_accuracy'], label='CNN')
plt.plot(history_mobilenet.history['val_accuracy'], label='MobileNet')
plt.plot(history_efficientnet.history['val_accuracy'], label='EfficientNet')

plt.title("Model Comparison (Validation Accuracy)")
plt.xlabel("Epochs")
plt.ylabel("Val Accuracy")
plt.legend()
plt.grid()

plt.show()

In [ ]:
import pandas as pd
import os
import numpy as np
from tensorflow.keras.preprocessing import image

df = pd.read_csv("/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/test_solution_01.csv")

ids = df['ID'].tolist() 

In [ ]:
test_dir = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final"

In [ ]:
import numpy as np

images = []
valid_ids = []

for img_name in ids:
    img_path = os.path.join(test_dir, img_name)

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img) / 255.0

    images.append(img_array)
    valid_ids.append(img_name)

# Convert to array
images = np.array(images)

# 🔥 SINGLE PREDICTION CALL
preds = best_model.predict(images)

# Convert to labels
preds = (preds > 0.5).astype(int).reshape(-1)

In [ ]:
# reverse mapping
labels_map = {0: 'chihuahua', 1: 'muffin'}

preds = (preds > 0.5).astype(int).reshape(-1)

pred_labels = [labels_map[p] for p in preds]

In [ ]:
submission = pd.DataFrame({
    "ID": valid_ids,
    "Label": pred_labels
})

submission.to_csv("submission.csv", index=False)

In [ ]:
print(train_data.class_indices)

In [ ]:
submission.head(20)
#Itz Ary99Pk